# z719 final gerencial, futuro = 202109
Francisco Medina Raed. Basado en el z719 de la cátedra, con lo que salió del Experimento Colaborativo 07 (Grupo A) y tres mejoras propias.

**Qué cambia respecto del z719 original**

| | z719 original | Este notebook |
|---|---|---|
| Training strategy | 202005 a 202106 (todos los meses) | Dos variantes del Exp 07: **desde 202103** (ganadora) y **desde 202101** (prudente) |
| Final train | 202005 a 202107 | 202103 a 202107 y 202101 a 202107 |
| Modelo final | `c(fijos, mejores)`: ignora el grid search | `modifyList(fijos, mejores)` |
| Grid search | 4 × 5 = 20 combinaciones | **5 × 7 = 35**: suma `num_leaves` 1024 y `min_data_in_leaf` 16 y 32 (en el Exp 07 el óptimo caía en el borde) |
| Modelo final | 1 semilla | **Semillerío de 20 semillas**, promediando las probabilidades |
| Cortes | 800 a 1.300 de a 50 | **800 a 1.400 de a 50** (13 por variante, 26 en total) |

Todo lo demás (catastrophe analysis, FE intra-mes, lags y deltas, parámetros fijos de LightGBM, AUC con early stopping) es idéntico al z719.

**Cómo correrlo (VM de Google Cloud, Jupyter con kernel R)**
1. Completar `PARAM$semilla_primigenia` en la primera celda.
2. Opcional: correr primero con `modo <- "test"` (grilla chica, 3 semillas, sin subir a Kaggle) para verificar que todo anda.
3. Con `modo <- "full"`, correr las celdas en orden. Tarda varias horas: 35 combinaciones de grid y 20 modelos finales por variante.
4. **Es reanudable**: si la VM se corta, se vuelve a correr y saltea las combinaciones del grid y las semillas que ya terminaron. Las subidas a Kaggle que fallen (por ejemplo, por el cupo diario) quedan en `pendientes_kaggle.txt` y se reintentan al volver a correr.
5. La última celda baja los puntajes Public, arma la curva de ganancia y sugiere el corte para la entrega final.

Necesita `~/.kaggle/kaggle.json` en la VM. Todo queda en `~/buckets/b1/exp/WF7190/`.

## Parámetros, paquetes y carpetas

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

modo <- "full"           # "test" = corrida corta para probar que todo anda (no sube a Kaggle)
subir_kaggle <- TRUE     # sube los archivos a Kaggle apenas se generan (necesita el kaggle.json)

PARAM <- list()
PARAM$semilla_primigenia <- 123457     # <- tu semilla primigenia
PARAM$experimento <- 7190
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# --- Training strategy: lo que salio del Experimento 07 (Grupo A) ---
# Se corren las dos mejores variantes y se comparan en Kaggle
PARAM$variantes <- list(
  list(nombre = "V3_desde202103", desde = 202103,
       descripcion = "ganadora del Exp 07: training 202103-202106, final_train 202103-202107"),
  list(nombre = "V2_desde202101", desde = 202101,
       descripcion = "opcion prudente del Exp 07: training 202101-202106, final_train 202101-202107")
)
PARAM$trainingstrategy$validate     <- 202107
PARAM$trainingstrategy$hasta        <- 202106   # ultimo mes de training
PARAM$trainingstrategy$future       <- 202109
PARAM$trainingstrategy$training_pct <- 1.0      # sin undersampling, como el z719
PARAM$trainingstrategy$positivos    <- c("BAJA+1", "BAJA+2")

# --- Grid search ampliado: en el Exp 07 el optimo caia en el borde (min_data_in_leaf = 64) ---
PARAM$grid <- list(
  num_leaves       = c(64, 128, 256, 512, 1024),
  min_data_in_leaf = c(16, 32, 64, 256, 512, 1024, 2048)
)

# --- Parametros fijos de LightGBM: identicos al z719 ---
PARAM$lgbm$param_fijos <- list(
  objective = "binary",
  metric = "auc",
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = PARAM$semilla_primigenia,
  max_bin = 31,
  learning_rate = 0.03,
  feature_fraction = 0.5,
  num_iterations = 2048,        # lo limita el early stopping
  early_stopping_rounds = 200,
  num_leaves = 64,
  min_data_in_leaf = 128
)

# --- Semillerio: el modelo final se entrena con N semillas y se promedian las probabilidades ---
PARAM$semillerio$cantidad <- 20

# --- Kaggle: cortes de a 50 alrededor del pico del Exp 07 (unos 1.100 envios) ---
PARAM$kaggle$competencia <- "labo-1-virtual-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1400, by = 50)

if (modo == "test") {
  PARAM$experimento <- paste0(PARAM$experimento, "_test")
  PARAM$grid <- list(num_leaves = c(64, 256), min_data_in_leaf = c(64, 512))
  PARAM$lgbm$param_fijos$num_iterations <- 60
  PARAM$lgbm$param_fijos$early_stopping_rounds <- 20
  PARAM$semillerio$cantidad <- 3
  subir_kaggle <- FALSE
}

# semillas del semillerio: la primera es la primigenia, el resto se derivan de ella (reproducible)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillerio$semillas <- c(PARAM$semilla_primigenia,
  setdiff(sample(100000:999999, PARAM$semillerio$cantidad + 5), PARAM$semilla_primigenia)[seq_len(PARAM$semillerio$cantidad - 1)])

for (paquete in c("data.table", "R.utils", "lightgbm", "yaml")) {
  if (!require(paquete, character.only = TRUE)) {
    install.packages(paquete)
    require(paquete, character.only = TRUE)
  }
}

# carpeta del experimento en el bucket de la VM (o en el home si no hay bucket)
base <- path.expand(if (dir.exists("~/buckets/b1")) "~/buckets/b1/exp" else "~/exp")
carpeta_exp <- file.path(base, paste0("WF", PARAM$experimento))
dir.create(carpeta_exp, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta_exp)

# dataset: lo busco donde lo suele dejar la VM; si no esta, lo descargo
candidatos <- path.expand(file.path(c("~/datasets", "~/buckets/b1/datasets", carpeta_exp), PARAM$dataset))
archivo_dataset <- candidatos[file.exists(candidatos)][1]
if (is.na(archivo_dataset)) {
  archivo_dataset <- file.path(carpeta_exp, PARAM$dataset)
  download.file(paste0("https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/", PARAM$dataset),
                archivo_dataset, mode = "wb")
}

message("carpeta: ", carpeta_exp, "\ndataset: ", archivo_dataset,
        "\nsemillas del semillerio: ", paste(PARAM$semillerio$semillas, collapse = ", "))

## 6.3.1 Preprocesamiento
Idéntico al z719: catastrophe analysis de 202006, FE intra-mes y FE histórico (lags y deltas de orden 1 y 2). Se agrega solo una verificación de que las filas de cada cliente estén en orden cronológico, porque los lags dependen de eso.

In [ ]:
dataset <- fread(archivo_dataset)

# 6.3.1.2 Catastrophe Analysis (identico al z719)
dataset[foto_mes == 202006, internet := NA]
dataset[foto_mes == 202006, mrentabilidad := NA]
dataset[foto_mes == 202006, mrentabilidad_annual := NA]
dataset[foto_mes == 202006, mcomisiones := NA]
dataset[foto_mes == 202006, mactivos_margen := NA]
dataset[foto_mes == 202006, mpasivos_margen := NA]
dataset[foto_mes == 202006, mcuentas_saldo := NA]
dataset[foto_mes == 202006, ctarjeta_visa_transacciones := NA]
dataset[foto_mes == 202006, mtarjeta_visa_consumo := NA]
dataset[foto_mes == 202006, mtarjeta_master_consumo := NA]
dataset[foto_mes == 202006, ccallcenter_transacciones := NA]
dataset[foto_mes == 202006, chomebanking_transacciones := NA]

# 6.3.1.3 FE intra-mes (identico al z719)
atributos_presentes <- function(patributos) {
  atributos <- unique(patributos)
  length(atributos) == length(intersect(atributos, colnames(dataset)))
}
if (atributos_presentes(c("foto_mes")))
  dataset[, kmes := foto_mes %% 100]
if (atributos_presentes(c("mpayroll", "cliente_edad")))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# 6.3.1.5 FE historico: lags y deltas de orden 1 y 2 (identico al z719)
# los lags usan la fila anterior de cada cliente: verifico que esten en orden cronologico
stopifnot(dataset[, all(diff(foto_mes) > 0), by = numero_de_cliente][, all(V1)])

cols_lagueables <- copy(setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria")))
dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# clase binaria y azar para el (eventual) undersampling, como el z719
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1L, 0L)]
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar := runif(nrow(dataset))]

campos_buenos <- copy(setdiff(colnames(dataset),
  c("clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")))

message(format(Sys.time(), "%a %b %d %X %Y"), "  preprocesado: ",
        nrow(dataset), " filas x ", length(campos_buenos), " variables")

## Funciones: grid search, semillerío y Kaggle
- **`grid_search`**: entrena con los meses de la variante, valida en 202107 y guarda cada combinación apenas termina.
- **`semillerio`**: entrena el modelo final con los mejores hiperparámetros (con `modifyList`) una vez por semilla, predice 202109 y promedia las probabilidades. Guarda una predicción por semilla en `semillas/`.
- **`generar_kaggle`**: ordena por probabilidad promedio, genera un archivo por corte y lo sube.

In [ ]:
# ---------- Kaggle ----------
preparar_kaggle <- function() {
  if (!file.exists("~/.kaggle/kaggle.json"))
    stop("falta ~/.kaggle/kaggle.json en la VM")
  if (Sys.which("kaggle") == "")
    invisible(system("python3 -m pip install -q --user kaggle"))
}

# sube un archivo; si falla (cupo diario, red) lo anota en pendientes_kaggle.txt para reintentar.
# los que ya subieron quedan en subidos_kaggle.txt y no se vuelven a subir al re-correr
# si una subida falla, el resto de la corrida ya no intenta: todo queda pendiente para la proxima vez
kaggle_sin_cupo <- FALSE
subir_a_kaggle <- function(archivo, mensaje) {
  archivo <- normalizePath(archivo)
  if (file.exists(file.path(carpeta_exp, "subidos_kaggle.txt")) &&
      archivo %in% readLines(file.path(carpeta_exp, "subidos_kaggle.txt"))) return(TRUE)
  if (kaggle_sin_cupo) {
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
    return(FALSE)
  }
  salida <- tryCatch(
    suppressWarnings(system2("kaggle", c("competitions", "submit", "-c", PARAM$kaggle$competencia,
                                         "-f", shQuote(archivo), "-m", shQuote(mensaje)),
                             stdout = TRUE, stderr = TRUE)),
    error = function(e) conditionMessage(e))
  ok <- any(grepl("Successfully submitted", salida))
  message("    kaggle ", if (ok) "OK   " else "ERROR", "  ", basename(archivo),
          if (ok) "" else paste0("  ", substr(paste(trimws(salida), collapse = " "), 1, 300)))
  if (ok) {
    cat(archivo, "\n", sep = "", file = file.path(carpeta_exp, "subidos_kaggle.txt"), append = TRUE)
    Sys.sleep(30)
  } else {
    kaggle_sin_cupo <<- TRUE
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
  }
  ok
}

reintentar_pendientes <- function() {
  arch <- file.path(carpeta_exp, "pendientes_kaggle.txt")
  if (!subir_kaggle || !file.exists(arch)) return(invisible())
  pendientes <- unique(setdiff(trimws(readLines(arch)), ""))
  file.remove(arch)
  message(length(pendientes), " archivos pendientes de subir a Kaggle")
  for (p in pendientes) {          # si falla uno, los siguientes quedan anotados sin intentar
    partes <- strsplit(p, "|", fixed = TRUE)[[1]]
    subir_a_kaggle(partes[1], partes[2])
  }
}

# ---------- Training strategy ----------
meses_training <- function(desde) {
  meses <- sort(unique(dataset$foto_mes), decreasing = TRUE)
  meses[meses >= desde & meses <= PARAM$trainingstrategy$hasta]
}

lgb_dataset <- function(filtro) {
  lgb.Dataset(data = data.matrix(dataset[filtro, campos_buenos, with = FALSE]),
              label = dataset[filtro, clase01], free_raw_data = TRUE)
}

# ---------- 6.3.2.2 Grid search (reanudable: guarda cada combinacion apenas termina) ----------
grid_search <- function(variante, carpeta) {
  archivo <- file.path(carpeta, "tb_grid_search.txt")
  tb <- CJ(num_leaves = PARAM$grid$num_leaves, min_data_in_leaf = PARAM$grid$min_data_in_leaf)
  if (file.exists(archivo)) tb <- tb[!fread(archivo), on = .(num_leaves, min_data_in_leaf)]

  if (nrow(tb)) {
    training <- meses_training(variante$desde)
    filtro_train <- dataset[, foto_mes %in% training &
      (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]
    dtrain <- lgb_dataset(filtro_train)
    dvalidate <- lgb_dataset(dataset[, foto_mes %in% PARAM$trainingstrategy$validate])

    for (i in seq_len(nrow(tb))) {
      x <- as.list(tb[i])
      modelo <- lgb.train(data = dtrain, valids = list(valid = dvalidate), eval = "auc",
                          param = modifyList(PARAM$lgbm$param_fijos, x), verbose = -100)
      fila <- data.table(tb[i], AUC = modelo$record_evals$valid$auc$eval[[modelo$best_iter]],
                         num_iterations = modelo$best_iter)
      fwrite(fila, archivo, sep = "\t", append = file.exists(archivo))
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  nl=", x$num_leaves,
              " mdil=", x$min_data_in_leaf, "  niter ", fila$num_iterations, "  AUC ", round(fila$AUC, 5))
      rm(modelo); gc(full = TRUE, verbose = FALSE)
    }
    rm(dtrain, dvalidate); gc(full = TRUE, verbose = FALSE)
  }
  res <- fread(archivo)
  setorder(res, -AUC)
  res
}

# ---------- 6.3.3 Final training con semillerio (reanudable: una prediccion por semilla) ----------
semillerio <- function(variante, carpeta, mejores) {
  carpeta_sem <- file.path(carpeta, "semillas")
  dir.create(carpeta_sem, showWarnings = FALSE)

  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  # modifyList: los mejores del grid PISAN a los fijos (con c() quedaban duplicados y LightGBM usaba los fijos)
  param_final <- modifyList(fijos, mejores)

  pendientes <- PARAM$semillerio$semillas[
    !file.exists(file.path(carpeta_sem, paste0("pred_s", PARAM$semillerio$semillas, ".txt")))]

  if (length(pendientes)) {
    final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
    dfinal_train <- lgb_dataset(dataset[, foto_mes %in% final_train])
    dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
    mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

    for (s in pendientes) {
      param_final$seed <- s
      modelo <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)
      fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, prob = predict(modelo, mfuture)),
             file.path(carpeta_sem, paste0("pred_s", s, ".txt")), sep = "\t")
      if (s == PARAM$semilla_primigenia) {          # modelo e importancia solo de la semilla primigenia
        lgb.save(modelo, file.path(carpeta, "modelo.txt"))
        fwrite(as.data.table(lgb.importance(modelo)), file.path(carpeta, "impo.txt"), sep = "\t")
      }
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  semilla ", s, " lista")
      rm(modelo); gc(full = TRUE, verbose = FALSE)
    }
    rm(dfinal_train, dfuture, mfuture); gc(full = TRUE, verbose = FALSE)
  }

  # promedio de las probabilidades de todas las semillas
  preds <- rbindlist(lapply(PARAM$semillerio$semillas, function(s)
    fread(file.path(carpeta_sem, paste0("pred_s", s, ".txt")))[, semilla := s]))
  tb_prediccion <- preds[, .(prob = mean(prob)), by = numero_de_cliente]
  fwrite(tb_prediccion, file.path(carpeta, "prediccion.txt"), sep = "\t")   # util para ensembles
  list(prediccion = tb_prediccion, param_final = param_final)
}

# ---------- Archivos para Kaggle ----------
generar_kaggle <- function(variante, carpeta, tb_prediccion) {
  dir.create(file.path(carpeta, "kaggle"), showWarnings = FALSE)
  tb <- copy(tb_prediccion)
  setorder(tb, -prob)
  for (envios in PARAM$kaggle$cortes) {
    tb[, Predicted := 0L]
    tb[1:envios, Predicted := 1L]
    archivo <- file.path(carpeta, "kaggle",
                         paste0("KA", PARAM$experimento, "_", variante$nombre, "_", envios, ".csv"))
    fwrite(tb[, .(numero_de_cliente, Predicted)], archivo, sep = ",")
    if (subir_kaggle)
      subir_a_kaggle(archivo, paste0("z719 ", variante$nombre, " semillerio=", PARAM$semillerio$cantidad,
                                     " semilla=", PARAM$semilla_primigenia, " envios=", envios))
  }
}

## Corridas
Para cada variante: grid search → mejores hiperparámetros → semillerío → archivos y subida a Kaggle → `PARAM.yml`.

In [ ]:
if (subir_kaggle) { preparar_kaggle(); reintentar_pendientes() }
t0 <- Sys.time()
resumen <- list()

for (variante in PARAM$variantes) {
  carpeta <- file.path(carpeta_exp, variante$nombre)
  dir.create(carpeta, showWarnings = FALSE)
  message(format(Sys.time(), "%a %b %d %X %Y"), "  ===== ", variante$nombre,
          ": training desde ", variante$desde, " =====")

  # grid search
  tb_grid <- grid_search(variante, carpeta)
  mejores <- as.list(tb_grid[1, .(num_leaves, min_data_in_leaf, num_iterations)])
  message("  mejores: nl=", mejores$num_leaves, " mdil=", mejores$min_data_in_leaf,
          " niter=", mejores$num_iterations, "  AUC ", round(tb_grid[1, AUC], 5))
  if (mejores$min_data_in_leaf == min(PARAM$grid$min_data_in_leaf) ||
      mejores$num_leaves == max(PARAM$grid$num_leaves))
    warning(variante$nombre, ": el optimo cayo en el borde de la grilla ampliada")

  # final training con semillerio y prediccion de 202109
  fin <- semillerio(variante, carpeta, mejores)

  # archivos y subida a Kaggle
  generar_kaggle(variante, carpeta, fin$prediccion)

  # registro de la corrida
  PARAM_var <- PARAM
  PARAM_var$variante <- variante
  PARAM_var$trainingstrategy$training <- meses_training(variante$desde)
  PARAM_var$trainingstrategy$final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
  PARAM_var$out$lgbm$AUC <- tb_grid[1, AUC]
  PARAM_var$out$lgbm$mejores_hiperparametros <- mejores
  PARAM_var$out$param_final <- fin$param_final
  write_yaml(PARAM_var, file.path(carpeta, "PARAM.yml"))

  resumen[[variante$nombre]] <- data.table(variante = variante$nombre, desde = variante$desde,
    AUC_valid = tb_grid[1, AUC], num_leaves = mejores$num_leaves,
    min_data_in_leaf = mejores$min_data_in_leaf, num_iterations = mejores$num_iterations,
    semillas = PARAM$semillerio$cantidad)
}

reintentar_pendientes()
resumen <- rbindlist(resumen)
fwrite(resumen, "resumen_variantes.csv")
message(format(Sys.time(), "%a %b %d %X %Y"), "  terminado en ",
        round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " min")
resumen

## Puntajes de Kaggle y elección del corte
Baja los puntajes Public de los envíos de este notebook y muestra la curva de ganancia por corte para cada variante.

Para la entrega final no conviene elegir el pico aislado del Public, que puede ser ruido. Por eso se sugiere el corte cuya **ventana de 3 cortes vecinos** tiene el mejor promedio. Ese archivo es el que hay que **marcar como entrega final en Kaggle**.

In [ ]:
descargar_envios <- TRUE   # FALSE para reusar el envios_kaggle.csv ya bajado

if (descargar_envios) {
  preparar_kaggle()
  # la CLI lista hasta 200 envios: bajo la lista completa con la API de Python, pagina por pagina
  writeLines(c(
    "import csv, sys",
    "from kaggle.api.kaggle_api_extended import KaggleApi",
    "from kagglesdk.competitions.types.competition_api_service import ApiListSubmissionsRequest",
    "from kagglesdk.competitions.types.competition_enums import SubmissionGroup, SubmissionSortBy",
    "api = KaggleApi()",
    "api.authenticate()",
    "envios, token = [], ''",
    "for pagina in range(50):",
    "    with api.build_kaggle_client() as kaggle:",
    "        req = ApiListSubmissionsRequest()",
    "        req.competition_name = sys.argv[1]",
    "        req.group = SubmissionGroup.SUBMISSION_GROUP_ALL",
    "        req.sort_by = SubmissionSortBy.SUBMISSION_SORT_BY_DATE",
    "        api._set_paging(req, 200, token, -1)",
    "        resp = kaggle.competitions.competition_api_client.list_submissions(req)",
    "    envios.extend(resp.submissions or [])",
    "    token = getattr(resp, 'next_page_token', '') or ''",
    "    if not token or not resp.submissions:",
    "        break",
    "with open(sys.argv[2], 'w', newline='', encoding='utf-8') as f:",
    "    w = csv.writer(f)",
    "    w.writerow(['ref', 'archivo', 'descripcion', 'fecha', 'estado', 'public'])",
    "    for s in envios:",
    "        w.writerow([s.ref, s.file_name, s.description, s.date, s.status, s.public_score])",
    "print(len(envios), 'envios en la competencia')"
  ), "bajar_envios_kaggle.py")
  invisible(system(paste("python3 bajar_envios_kaggle.py", PARAM$kaggle$competencia, "envios_kaggle.csv")))
}

# me quedo con los envios de este notebook: KA<experimento>_<variante>_<envios>.csv
patron <- paste0("^KA", PARAM$experimento, "_(.+)_([0-9]+)\\.csv$")
envios_kaggle <- fread("envios_kaggle.csv", colClasses = "character")
envios_kaggle[, public := suppressWarnings(as.numeric(public))]
puntajes <- envios_kaggle[grepl(patron, archivo) & grepl("COMPLETE", estado) & !is.na(public)]
puntajes[, variante := sub(patron, "\\1", archivo)]
puntajes[, envios := as.integer(sub(patron, "\\2", archivo))]
setorder(puntajes, fecha)
puntajes <- puntajes[, .SD[.N], by = .(variante, envios)]   # si se subio dos veces, el mas reciente
fwrite(puntajes[, .(variante, envios, public, fecha, ref, archivo)], "puntajes_kaggle.csv")

# curva de ganancia Public por corte
curva <- dcast(puntajes, envios ~ variante, value.var = "public")
print(curva)

# el pico aislado del Public es ruido: elijo el corte con mejor promedio en una ventana de 3 cortes vecinos
setorder(puntajes, variante, envios)
puntajes[, ventana3 := frollmean(public, 3, align = "center"), by = variante]
mejor <- puntajes[!is.na(ventana3)][order(-ventana3)][1]
fwrite(puntajes, "puntajes_ventanas.csv")
message("Maximo Public: ", puntajes[which.max(public), paste0(variante, " en ", envios, " envios (", public, ")")])
message("Corte estable sugerido: ", mejor$variante, " en ", mejor$envios, " envios",
        "  (promedio de la ventana ", round(mejor$ventana3, 3), ", Public del corte ", mejor$public, ")")
message("Archivo a marcar como entrega final en Kaggle: ", mejor$archivo)

## V2 rescatada (min_data_in_leaf >= 64)
En la corrida completa, V2 eligió `min_data_in_leaf = 32`, un valor que solo existe en la grilla ampliada: tuvo mejor AUC en julio pero la ganancia Public se desplomó (de 11 a 18, contra unos 24 en el Exp 07).

Esta celda **no repite el grid search**: lee el `tb_grid_search.txt` de V2, toma la mejor combinación con `min_data_in_leaf >= 64` y rehace solo el semillerío y los 13 envíos a Kaggle, en la carpeta `V2r_mdil64_desde202101`. Requiere haber corrido antes las celdas 1, 2 y 3. Cuando Kaggle puntúe los envíos, volver a correr la celda de puntajes: la nueva variante aparece como una columna más.


In [ ]:
# ===== V2 rescatada: mejor combinacion del grid de V2 con min_data_in_leaf >= 64 =====
# Requiere haber corrido antes las celdas 1, 2 y 3 (parametros, preprocesamiento y funciones).
# No repite el grid search: lee el que ya hizo V2 y rehace solo el semillerio y los envios a Kaggle.

MDIL_MINIMO <- 64

variante_origen <- PARAM$variantes[[which(sapply(PARAM$variantes, `[[`, "nombre") == "V2_desde202101")]]
variante_rescate <- list(
  nombre = paste0("V2r_mdil", MDIL_MINIMO, "_desde202101"),
  desde = variante_origen$desde,
  descripcion = paste0("V2 con la mejor combinacion del grid restringida a min_data_in_leaf >= ", MDIL_MINIMO)
)

# grid de V2 ya corrido, ordenado por AUC
tb_grid_v2 <- fread(file.path(carpeta_exp, variante_origen$nombre, "tb_grid_search.txt"))
setorder(tb_grid_v2, -AUC)
message("Top 5 del grid de V2 sin restriccion:")
print(head(tb_grid_v2, 5))

tb_restringido <- tb_grid_v2[min_data_in_leaf >= MDIL_MINIMO]
message("\nTop 5 con min_data_in_leaf >= ", MDIL_MINIMO, ":")
print(head(tb_restringido, 5))

mejores <- as.list(tb_restringido[1, .(num_leaves, min_data_in_leaf, num_iterations)])
message("\nElegida: nl=", mejores$num_leaves, " mdil=", mejores$min_data_in_leaf,
        " niter=", mejores$num_iterations, "  AUC ", round(tb_restringido[1, AUC], 5),
        "  (la original de V2 tenia AUC ", round(tb_grid_v2[1, AUC], 5), ")")

carpeta <- file.path(carpeta_exp, variante_rescate$nombre)
dir.create(carpeta, showWarnings = FALSE)
fwrite(tb_restringido, file.path(carpeta, "tb_grid_search_restringido.txt"), sep = "\t")

if (subir_kaggle) { preparar_kaggle(); reintentar_pendientes() }

# semillerio con las mismas 20 semillas (reanudable) y envios a Kaggle
fin <- semillerio(variante_rescate, carpeta, mejores)
generar_kaggle(variante_rescate, carpeta, fin$prediccion)

PARAM_var <- PARAM
PARAM_var$variante <- variante_rescate
PARAM_var$grid_restriccion <- paste0("min_data_in_leaf >= ", MDIL_MINIMO, " sobre el grid de ", variante_origen$nombre)
PARAM_var$trainingstrategy$training <- meses_training(variante_rescate$desde)
PARAM_var$trainingstrategy$final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante_rescate$desde))
PARAM_var$out$lgbm$AUC <- tb_restringido[1, AUC]
PARAM_var$out$lgbm$mejores_hiperparametros <- mejores
PARAM_var$out$param_final <- fin$param_final
write_yaml(PARAM_var, file.path(carpeta, "PARAM.yml"))

# que tan distinta es la prediccion respecto de V2 original y de V3 (correlacion de rangos)
leer_pred <- function(v) fread(file.path(carpeta_exp, v, "prediccion.txt"))[, .(numero_de_cliente, prob)]
comp <- Reduce(function(a, b) merge(a, b, by = "numero_de_cliente"), list(
  leer_pred(variante_rescate$nombre)[, .(numero_de_cliente, V2r = prob)],
  leer_pred("V2_desde202101")[, .(numero_de_cliente, V2 = prob)],
  leer_pred("V3_desde202103")[, .(numero_de_cliente, V3 = prob)]))
message("\nCorrelacion de Spearman entre predicciones:")
print(round(cor(comp[, .(V2r, V2, V3)], method = "spearman"), 3))

reintentar_pendientes()
message(format(Sys.time(), "%a %b %d %X %Y"), "  listo: ", variante_rescate$nombre,
        ". Cuando Kaggle los puntue, correr de nuevo la celda de puntajes.")